<a href="https://colab.research.google.com/github/zannatulzerin/comp8240-goemotions-replication/blob/main/comp8240_goemotions_replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**GIT CLONING**

In [1]:
!git clone --filter=blob:none --sparse --depth 1 https://github.com/google-research/google-research.git gr_sparse
%cd gr_sparse
!git sparse-checkout set goemotions
%cd goemotions
!pwd

Cloning into 'gr_sparse'...
remote: Enumerating objects: 3312, done.
remote: Counting objects: 100% (3312/3312), done.
remote: Compressing objects: 100% (2706/2706), done.
remote: Total 3312 (delta 54), reused 2666 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (3312/3312), 822.19 KiB | 1.33 MiB/s, done.
Resolving deltas: 100% (54/54), done.
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 1), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 6.13 KiB | 1.53 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/gr_sparse
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 24 (delta 5), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 2.22 MiB | 2.82 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Updating files: 100% (26/26

**LOADING GOEMOTIONS DATASET**

In [2]:
!mkdir -p data/full_dataset
!wget -q -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv
!wget -q -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv
!wget -q -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_3.csv
!ls data/full_dataset/

goemotions_1.csv  goemotions_2.csv  goemotions_3.csv  README.md


In [3]:
!pip install -q absl-py statsmodels seaborn

**Running paper's analyze_data.py**

In [4]:
!python3 -m analyze_data --data=data/full_dataset

Loading data...
58011 Examples
211225 Annotations
28 emotion Categories
82 unique raters
0.016 marked unclear
Distribution of number of labels per example:
1     0.560703
2     0.416165
3     0.020297
4     0.001920
5     0.000510
6     0.000255
7     0.000096
8     0.000029
9     0.000014
10    0.000005
12    0.000005
Name: count, dtype: float64
0.00 with more than 3 labels
Label distributions:
neutral           26.61
neutral           26.61
approval           8.48
admiration         8.24
annoyance          6.55
gratitude          5.59
disapproval        5.50
curiosity          4.66
amusement          4.45
realization        4.23
optimism           4.19
disappointment     4.08
love               3.94
anger              3.89
joy                3.84
confusion          3.54
sadness            3.25
caring             2.89
excitement         2.71
surprise           2.65
disgust            2.55
desire             1.84
fear               1.54
remorse            1.22
embarrassment      1.19
n

**RoBERTa Classifier Evaluation**

In [5]:
!pip install -q transformers datasets torch

In [6]:
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import f1_score, classification_report
import numpy as np

# 1. Load test split + label names
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
test = dataset["test"]
label_names = test.features["labels"].feature.names
num_labels = len(label_names)

print(f"Test set size: {len(test)}")
print(f"Labels ({num_labels}): {label_names}")

# 2. Load classifier
classifier = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None,
    truncation=True,
)

# 3. DEBUG CHECK
sample_output = classifier(["I can't believe how good this is!"])
print("\n--- DEBUG: raw model output ---")
print(sample_output)
print("--- Does this match dataset label names? ---")
print(label_names[:5], "...")

# 4. Subsample for a first pass
SAMPLE_SIZE = 200
texts = test["text"][:SAMPLE_SIZE]
true_label_ids = test["labels"][:SAMPLE_SIZE]

y_true = np.zeros((len(texts), num_labels), dtype=int)
for i, ids in enumerate(true_label_ids):
    for label_id in ids:
        y_true[i, label_id] = 1

THRESHOLD = 0.5
y_pred = np.zeros((len(texts), num_labels), dtype=int)
results = classifier(texts, batch_size=16)

matched_count = 0
for i, example_scores in enumerate(results):
    for entry in example_scores:
        if entry["label"] in label_names and entry["score"] >= THRESHOLD:
            idx = label_names.index(entry["label"])
            y_pred[i, idx] = 1
            matched_count += 1

print(f"\nTotal label matches found across all examples: {matched_count}")
print(f"(If this is 0 or near 0, the label names don't match -- see debug output above)")

macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)

print(f"\nMacro F1: {macro_f1:.4f}")
print(f"Micro F1: {micro_f1:.4f}")
print("\nPer-label breakdown:")
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]


--- DEBUG: raw model output ---
[[{'label': 'admiration', 'score': 0.6253014802932739}, {'label': 'surprise', 'score': 0.5066424012184143}, {'label': 'excitement', 'score': 0.13753469288349152}, {'label': 'approval', 'score': 0.019480424001812935}, {'label': 'neutral', 'score': 0.018119733780622482}, {'label': 'curiosity', 'score': 0.017776543274521828}, {'label': 'joy', 'score': 0.011109255254268646}, {'label': 'realization', 'score': 0.010974415577948093}, {'label': 'love', 'score': 0.00929083488881588}, {'label': 'annoyance', 'score': 0.005031626205891371}, {'label': 'disapproval', 'score': 0.005010381806641817}, {'label': 'amusement', 'score': 0.004871685523539782}, {'label': 'disappointment', 'score': 0.004831920377910137}, {'label': 'confusion', 'score': 0.004518175031989813}, {'label': 'optimism', 'score': 0.0038963453844189644}, {'label': 'pride', 'score': 0.0029688309878110886}, {'label': 'anger', 'score': 0.0028320623096078634}, {'label': 'gratitude', 'score': 0.002580986358

**Running paper's ppca_py script**

In [7]:
with open("ppca.py", "r") as f:
    content = f.read()

content = content.replace(
    "pd.Series.nonzero(x)[0]",
    "x.to_numpy().nonzero()[0]"
)

with open("ppca.py", "w") as f:
    f.write(content)

print("Patched ppca.py successfully")

Patched ppca.py successfully


In [8]:
!python3 -m ppca --data=data/full_dataset

Loading data...
58011 Examples
211225 Annotations
28 emotion Categories
Processing data...
Ratings shape (36796, 5, 28)
Calculating leave-out (partial) correlations...
/content/gr_sparse/goemotions/ppca.py:115: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(res_x, res_y)
/content/gr_sparse/goemotions/ppca.py:208: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  c = spearmanr(left_out_scores[:, i], insample_scores[:, i])[0]
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:1006: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return fun(*args, **kwargs)
Which dimensions to keep?
[False]
[nan]
0.05
Running PPCA on all the data...
Plotting percentage of covariance explained...
/content/gr_sparse/goemotions/ppca.py:133: RuntimeWarning: divide by zero encountered in scalar 

**Testing Classifier on Amazon Fine Food Reviews dataset**

In [9]:
!wget -q https://snap.stanford.edu/data/finefoods.txt.gz
!gunzip -k finefoods.txt.gz
!ls -lh finefoods.txt

-rw-r--r-- 1 root root 354M Mar  5  2013 finefoods.txt


In [10]:
import pandas as pd

def parse_finefoods(path, max_records=None):
    records = []
    current = {}
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            line = line.strip()
            if not line:
                if current:
                    records.append(current)
                    current = {}
                if max_records and len(records) >= max_records:
                    break
                continue
            if ": " in line:
                key, _, value = line.partition(": ")
                current[key] = value
    if current:
        records.append(current)
    return pd.DataFrame(records)

df = parse_finefoods("finefoods.txt", max_records=50000)
print(df.shape)
df.head()

(50000, 8)


,product/productId,review/userId,review/profileName,review/helpfulness,review/score,review/time,review/summary,review/text
0,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1/1,5.0,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0/0,1.0,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1/1,4.0,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,B000UA0QIQ,A395BORC6FGVXV,Karl,3/3,2.0,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0/0,5.0,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [11]:
# Clean up column names
df = df.rename(columns={
    "product/productId": "ProductId",
    "review/userId": "UserId",
    "review/score": "Score",
    "review/summary": "Summary",
    "review/text": "Text",
})

print("Shape:", df.shape)
print("\nScore distribution:")
print(df["Score"].value_counts().sort_index())

print("\nSample review:")
print(df[["ProductId", "Score", "Summary", "Text"]].iloc[0])

df["review_length"] = df["Text"].str.split().str.len()
print("\nReview length stats (words):")
print(df["review_length"].describe())

Shape: (50000, 8)

Score distribution:
Score
1.0     4721
2.0     2814
3.0     4047
4.0     7288
5.0    31130
Name: count, dtype: int64

Sample review:
ProductId                                           B001E4KFG0
Score                                                      5.0
Summary                                  Good Quality Dog Food
Text         I have bought several of the Vitality canned d...
Name: 0, dtype: object

Review length stats (words):
count    50000.000000
mean        79.383860
std         75.430186
min          6.000000
25%         34.000000
50%         57.000000
75%         97.000000
max       1751.000000
Name: review_length, dtype: float64


In [12]:
# Reuse existing classifier
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=3,
    truncation=True,
)

sample_reviews = df["Text"].head(5).tolist()

for i, review in enumerate(sample_reviews):
    print(f"\n--- Review {i+1} ---")
    print(review[:150], "...")
    predictions = classifier(review)
    print("Top emotions:", predictions)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Review 1 ---
I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than  ...
Top emotions: [[{'label': 'admiration', 'score': 0.8027827739715576}, {'label': 'approval', 'score': 0.23201344907283783}, {'label': 'neutral', 'score': 0.09838002175092697}]]

--- Review 2 ---
Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor inten ...
Top emotions: [[{'label': 'confusion', 'score': 0.9001879096031189}, {'label': 'neutral', 'score': 0.17657306790351868}, {'label': 'curiosity', 'score': 0.028977045789361}]]

--- Review 3 ---
This is a confection that has been around a few centuries.  It is a light, pillowy citrus gelatin with nuts - in this case Filberts. And it is cut int ...
Top emotions: [[{'label': 'admiration', 'score': 0.4042718708515167}, {'label': 'approval', 'score': 0.3922702968120575}, {'label':

In [13]:
import pandas as pd
import numpy as np

# Define rough positive/negative emotion groups from the GoEmotions taxonomy
POSITIVE_EMOTIONS = {"admiration", "amusement", "approval", "caring", "desire",
                      "excitement", "gratitude", "joy", "love", "optimism", "pride", "relief"}
NEGATIVE_EMOTIONS = {"anger", "annoyance", "disappointment", "disapproval", "disgust",
                      "embarrassment", "fear", "grief", "nervousness", "remorse", "sadness"}

SAMPLE_SIZE = 100
sample_df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

results = classifier(sample_df["Text"].str.slice(0, 500).tolist(), batch_size=16)  # truncate very long reviews

records = []
for i, scores in enumerate(results):
    top_label = max(scores, key=lambda x: x["score"])["label"]
    pos_score = sum(s["score"] for s in scores if s["label"] in POSITIVE_EMOTIONS)
    neg_score = sum(s["score"] for s in scores if s["label"] in NEGATIVE_EMOTIONS)
    records.append({
        "Score": sample_df["Score"].iloc[i],
        "top_emotion": top_label,
        "positive_emotion_sum": pos_score,
        "negative_emotion_sum": neg_score,
    })

results_df = pd.DataFrame(records)

print("Average positive/negative emotion mass by star rating:")
print(results_df.groupby("Score")[["positive_emotion_sum", "negative_emotion_sum"]].mean())

print("\nMost common top-predicted emotion by star rating:")
print(results_df.groupby("Score")["top_emotion"].agg(lambda x: x.value_counts().index[0]))

print("\nA few concrete examples:")
for i in range(5):
    print(f"\nScore={sample_df['Score'].iloc[i]} | Top emotion={records[i]['top_emotion']}")
    print(sample_df["Text"].iloc[i][:150], "...")

Average positive/negative emotion mass by star rating:
       positive_emotion_sum  negative_emotion_sum
Score                                            
1.0                0.087243              0.496049
2.0                0.444702              0.247937
3.0                0.579284              0.132711
4.0                0.646170              0.067336
5.0                0.967488              0.017743

Most common top-predicted emotion by star rating:
Score
1.0       neutral
2.0          love
3.0       neutral
4.0    admiration
5.0    admiration
Name: top_emotion, dtype: object

A few concrete examples:

Score=5.0 | Top emotion=love
This is one of our favorite blends. The k-cups themselves seem to be well-made (no exploders--least so far). Italian is between medium and strong. It' ...

Score=3.0 | Top emotion=disapproval
The coffee was more on the light side rather than medium roast.  I don't think I would purchase it again even though it was a good value purchase. ...

Score=5.0 | Top

**Testing Classifier on Coronavirus Tweets**

In [14]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_a8a97caf51dd87c5de67def72913dae4'

In [15]:
!kaggle datasets download -d datatattle/covid-19-nlp-text-classification
!unzip -o covid-19-nlp-text-classification.zip
!ls *.csv

Dataset URL: https://www.kaggle.com/datasets/datatattle/covid-19-nlp-text-classification
License(s): copyright-authors
100% 4.38M/4.38M [00:01<00:00, 3.41MB/s]

Archive:  covid-19-nlp-text-classification.zip
  inflating: Corona_NLP_test.csv     
  inflating: Corona_NLP_train.csv    
Corona_NLP_test.csv  Corona_NLP_train.csv


In [16]:
import pandas as pd

train_df = pd.read_csv("Corona_NLP_train.csv", encoding="latin-1")
test_df = pd.read_csv("Corona_NLP_test.csv", encoding="latin-1")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nColumns:", list(train_df.columns))
train_df.head()

Train shape: (41157, 6)
Test shape: (3798, 6)

Columns: ['UserName', 'ScreenName', 'Location', 'TweetAt', 'OriginalTweet', 'Sentiment']


,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative


In [17]:
print("Sentiment label distribution:")
print(train_df["Sentiment"].value_counts())

print("\nSample tweet:")
print(train_df[["OriginalTweet", "Sentiment", "TweetAt"]].iloc[0])

train_df["tweet_length"] = train_df["OriginalTweet"].str.split().str.len()
print("\nTweet length stats (words):")
print(train_df["tweet_length"].describe())

print("\nDate range:", train_df["TweetAt"].min(), "to", train_df["TweetAt"].max())

Sentiment label distribution:
Sentiment
Positive              11422
Negative               9917
Neutral                7713
Extremely Positive     6624
Extremely Negative     5481
Name: count, dtype: int64

Sample tweet:
OriginalTweet    @MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...
Sentiment                                                  Neutral
TweetAt                                                 16-03-2020
Name: 0, dtype: object

Tweet length stats (words):
count    41157.000000
mean        30.500304
std         11.591172
min          1.000000
25%         21.000000
50%         32.000000
75%         40.000000
max         64.000000
Name: tweet_length, dtype: float64

Date range: 01-04-2020 to 31-03-2020


In [18]:
SAMPLE_SIZE = 100
sample_tweets_df = train_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

results = classifier(sample_tweets_df["OriginalTweet"].str.slice(0, 500).tolist(), batch_size=16)

records = []
for i, scores in enumerate(results):
    top_label = max(scores, key=lambda x: x["score"])["label"]
    pos_score = sum(s["score"] for s in scores if s["label"] in POSITIVE_EMOTIONS)
    neg_score = sum(s["score"] for s in scores if s["label"] in NEGATIVE_EMOTIONS)
    records.append({
        "Sentiment": sample_tweets_df["Sentiment"].iloc[i],
        "top_emotion": top_label,
        "positive_emotion_sum": pos_score,
        "negative_emotion_sum": neg_score,
    })

results_df = pd.DataFrame(records)

print("Average positive/negative emotion mass by Sentiment label:")
print(results_df.groupby("Sentiment")[["positive_emotion_sum", "negative_emotion_sum"]].mean())

print("\nMost common top-predicted emotion by Sentiment label:")
print(results_df.groupby("Sentiment")["top_emotion"].agg(lambda x: x.value_counts().index[0]))

print("\nA few concrete examples:")
for i in range(5):
    print(f"\nSentiment={sample_tweets_df['Sentiment'].iloc[i]} | Top emotion={records[i]['top_emotion']}")
    print(sample_tweets_df["OriginalTweet"].iloc[i][:150], "...")

Average positive/negative emotion mass by Sentiment label:
                    positive_emotion_sum  negative_emotion_sum
Sentiment                                                     
Extremely Negative              0.115380              0.133099
Extremely Positive              0.338468              0.004233
Negative                        0.160245              0.132719
Neutral                         0.023564              0.079721
Positive                        0.240065              0.071829

Most common top-predicted emotion by Sentiment label:
Sentiment
Extremely Negative    neutral
Extremely Positive    neutral
Negative              neutral
Neutral               neutral
Positive              neutral
Name: top_emotion, dtype: object

A few concrete examples:

Sentiment=Neutral | Top emotion=nervousness
 Without the there would not be any problem whatsoever People are getting worried about the supply chain   Prices for key food staples are starting to ...

Sentiment=Extremely Negat

**Constructing New Dataset (Ongoing)**

In [19]:
from datasets import load_dataset

aita_data = load_dataset("derek-thomas/dataset-creator-reddit-amitheasshole")
aita_df = aita_data["train"].to_pandas()
print(aita_df.shape)
aita_df.head()

README.md:   0%|          | 0.00/2.80k [00:00<?, ?B/s]

data/train-00000-of-00001-80f54f12efd954(…): reconstructing file:   0%|          |  0.00B / 3.20MB            

data/train-00000-of-00001-80f54f12efd954(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2570 [00:00<?, ? examples/s]

(2570, 9)


,content,poster,date_utc,flair,title,permalink,id,content_length,score
0,my boyfriend and I live with my parents. I pay...,Adventurous_Cat2691,2023-10-26 22:36:52,None,AITA For buying and making my own food when I ...,/r/AmItheAsshole/comments/17h8qys/aita_for_buy...,17h8qys,1308,6
1,Hey Reddit! My original post got taken down fo...,EdgucatedCheerful,2023-10-26 22:41:17,None,AITA for following the rules in highschool gym...,/r/AmItheAsshole/comments/17h8ui0/aita_for_fol...,17h8ui0,2862,0
2,"\nTo provide context, our daughter is 19 month...",Character-Bus8261,2023-10-26 22:46:10,None,AITA- telling daughter to swear in wife’s lang...,/r/AmItheAsshole/comments/17h8y9x/aita_telling...,17h8y9x,2156,1
3,My husband sprung a very sudden trip on us to ...,MemoryOk8893,2023-10-26 22:46:29,None,WIBTA if I ruined the surprise?,/r/AmItheAsshole/comments/17h8yhi/wibta_if_i_r...,17h8yhi,1317,3
4,I'm a little on the fence about this although ...,PercentageWide6608,2023-10-26 22:53:16,None,AITA for quitting my job without notice?,/r/AmItheAsshole/comments/17h93q8/aita_for_qui...,17h93q8,3406,7


In [20]:
!kaggle datasets download -d entenam/reddit-mental-health-dataset
!unzip -o reddit-mental-health-dataset.zip -d rmhd_data
!find rmhd_data -name "*.csv" | head -5

Dataset URL: https://www.kaggle.com/datasets/entenam/reddit-mental-health-dataset
License(s): CC0-1.0
100% 617M/617M [00:29<00:00, 21.9MB/s]

Archive:  reddit-mental-health-dataset.zip
  inflating: rmhd_data/Original Reddit Data/Labelled Data/LD DA 1.csv  
  inflating: rmhd_data/Original Reddit Data/Labelled Data/LD EL1.csv  
  inflating: rmhd_data/Original Reddit Data/Labelled Data/LD PF1.csv  
  inflating: rmhd_data/Original Reddit Data/Labelled Data/LD TS 1.csv  
  inflating: rmhd_data/Original Reddit Data/Labelled Data/Meta Labelled data kaggel.rtf  
  inflating: rmhd_data/Original Reddit Data/raw data/2019/Aug/anipjaug19.csv  
  inflating: rmhd_data/Original Reddit Data/raw data/2019/Aug/depjaug19.csv  
  inflating: rmhd_data/Original Reddit Data/raw data/2019/Aug/lonaug19.csv  
  inflating: rmhd_data/Original Reddit Data/raw data/2019/Aug/mhaug19.csv  
  inflating: rmhd_data/Original Reddit Data/raw data/2019/Aug/swaug19.csv  
  inflating: rmhd_data/Original Reddit Data/raw data/

In [21]:
print(aita_df.columns.tolist())
aita_df.head(3)

['content', 'poster', 'date_utc', 'flair', 'title', 'permalink', 'id', 'content_length', 'score']


,content,poster,date_utc,flair,title,permalink,id,content_length,score
0,my boyfriend and I live with my parents. I pay...,Adventurous_Cat2691,2023-10-26 22:36:52,None,AITA For buying and making my own food when I ...,/r/AmItheAsshole/comments/17h8qys/aita_for_buy...,17h8qys,1308,6
1,Hey Reddit! My original post got taken down fo...,EdgucatedCheerful,2023-10-26 22:41:17,None,AITA for following the rules in highschool gym...,/r/AmItheAsshole/comments/17h8ui0/aita_for_fol...,17h8ui0,2862,0
2,"\nTo provide context, our daughter is 19 month...",Character-Bus8261,2023-10-26 22:46:10,None,AITA- telling daughter to swear in wife’s lang...,/r/AmItheAsshole/comments/17h8y9x/aita_telling...,17h8y9x,2156,1


In [22]:
import pandas as pd

new_reddit_comments = aita_df.rename(columns={"content": "text"})[["text", "date_utc", "score"]].copy()
new_reddit_comments["subreddit"] = "AmItheAsshole"

new_reddit_comments["text"] = new_reddit_comments["text"].str.strip()
new_reddit_comments = new_reddit_comments.drop_duplicates(subset="text")
new_reddit_comments["word_count"] = new_reddit_comments["text"].str.split().str.len()
new_reddit_comments = new_reddit_comments[new_reddit_comments["word_count"] >= 3]

print(f"Final dataset: {len(new_reddit_comments)} comments")
new_reddit_comments.to_csv("new_reddit_comments.csv", index=False)
new_reddit_comments.sample(5)

Final dataset: 2567 comments


,text,date_utc,score,subreddit,word_count
913,So I (28m) have a friend “Andrew” (26m) who I’...,2023-11-05 15:29:16,5,AmItheAsshole,339
458,We are planning a trip out of state in a coupl...,2023-10-30 02:16:40,14,AmItheAsshole,122
2002,So I’ve been having a online relationship with...,2023-11-18 05:45:10,2,AmItheAsshole,541
2098,"Hey guys,\n\ni live in a family of 5 where my ...",2023-11-18 14:25:41,244,AmItheAsshole,377
804,I have never smoked and I heavily dislike the ...,2023-11-05 04:29:26,0,AmItheAsshole,359


In [23]:
print("Earliest post:", new_reddit_comments["date_utc"].min())
print("Latest post:", new_reddit_comments["date_utc"].max())

Earliest post: 2023-10-26 22:36:52
Latest post: 2023-12-04 11:46:34


In [24]:
from huggingface_hub import list_datasets

results = list_datasets(search="reddit amitheasshole", limit=10)
for r in results:
    print(r.id)

derek-thomas/dataset-creator-reddit-amitheasshole
Oguzz07/reddit-amitheasshole-dataset
sbussiso/AmItheAsshole-reddit-9-29-2025


In [25]:
from datasets import load_dataset

aita_data = load_dataset("derek-thomas/dataset-creator-reddit-amitheasshole")
aita_df = aita_data["train"].to_pandas()
print(aita_df.shape)

(2570, 9)


In [26]:
import pandas as pd

new_reddit_comments = aita_df.rename(columns={"content": "text"})[["text", "date_utc", "score"]].copy()
new_reddit_comments["subreddit"] = "AmItheAsshole"
new_reddit_comments["text"] = new_reddit_comments["text"].str.strip()
new_reddit_comments = new_reddit_comments.drop_duplicates(subset="text")
new_reddit_comments["word_count"] = new_reddit_comments["text"].str.split().str.len()
new_reddit_comments = new_reddit_comments[new_reddit_comments["word_count"] >= 5]  # a bit stricter, cleaner examples

sample_150 = new_reddit_comments.sample(n=150, random_state=42).reset_index(drop=True)
sample_150.to_csv("new_reddit_comments.csv", index=False)
print(f"Constructed dataset: {len(sample_150)} comments")
sample_150.head(3)

Constructed dataset: 150 comments


,text,date_utc,score,subreddit,word_count
0,Alright I'll keep it as short and to the point...,2023-11-08 20:37:24,176,AmItheAsshole,274
1,Me (m26) and gf (f21) have been together for a...,2023-11-18 13:38:27,7,AmItheAsshole,451
2,"Hello, this happened 2 days ago. I (24F) have ...",2023-11-18 07:18:39,2,AmItheAsshole,556


In [27]:
GOEMOTIONS_LABELS = [
    "admiration","amusement","anger","annoyance","approval","caring","confusion",
    "curiosity","desire","disappointment","disapproval","disgust","embarrassment",
    "excitement","fear","gratitude","grief","joy","love","nervousness","optimism",
    "pride","realization","relief","remorse","sadness","surprise","neutral"
]

to_label = sample_150.sample(n=15, random_state=1).reset_index(drop=True)

manual_labels = []
for i, row in to_label.iterrows():
    print(f"\n[{i+1}/15] {row['text']}")
    print("-" * 80)
    label = input("Emotion: ")
    manual_labels.append(label.strip().lower())

to_label["manual_label"] = manual_labels
to_label.to_csv("manually_labeled_sample.csv", index=False)
print(to_label[["text", "manual_label"]])


[1/15] So over the past few months I’ve been living with 3 roommates in a shared house, each with our own private rooms. None of us really knew each other beforehand, and for the most part we keep to ourselves aside from some occasional interactions. I’ve been working almost 60 hour weeks, so when I’m not working, I’m usually out spending time with family, friends, or playing a video game in my room.

Our place came unfurnished, and one of the roommates, John, bought all of furniture for the living room (e.g. couch, rugs, tables, etc.). I’m pretty much down to split the cost for things like this, but he didn’t consult anyone beforehand, he just bought everything himself. In fact it’s a little bit overbearing with how much John has added without asking anyone, like running ethernet wiring across the house and putting his cat’s box (that never gets cleaned) in the living room. The only things in the living room of mine are my bike, my TV, a spare desk with a couple monitors (that I told

In [28]:
print("Label each comment with exactly one emotion from this list: " + ", ".join(GOEMOTIONS_LABELS))
print("Reply with a numbered list, one emotion per line.\n")
for i, text in enumerate(to_label["text"]):
    print(f"{i+1}. {text}")
    print()

Label each comment with exactly one emotion from this list: admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire, disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief, joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, surprise, neutral
Reply with a numbered list, one emotion per line.

1. So over the past few months I’ve been living with 3 roommates in a shared house, each with our own private rooms. None of us really knew each other beforehand, and for the most part we keep to ourselves aside from some occasional interactions. I’ve been working almost 60 hour weeks, so when I’m not working, I’m usually out spending time with family, friends, or playing a video game in my room.

Our place came unfurnished, and one of the roommates, John, bought all of furniture for the living room (e.g. couch, rugs, tables, etc.). I’m pretty much down to split the cost for things like this, but he didn’t

In [29]:
llm_labels = ["annoyance", "anger", "disappointment", "disapproval", "caring",
              "remorse", "caring", "annoyance", "sadness", "annoyance",
              "anger", "anger", "disgust", "annoyance", "amusement"]
to_label["llm_label"] = llm_labels
agreement = (to_label["manual_label"] == to_label["llm_label"]).mean()
print(f"Manual vs LLM agreement: {agreement:.0%}")
to_label[["text", "manual_label", "llm_label"]]

Manual vs LLM agreement: 73%


,text,manual_label,llm_label
0,So over the past few months I’ve been living w...,annoyance,annoyance
1,"&#x200B;\n\n... I'm tired of this female, to b...",anger,anger
2,My daughter’s school did not choose her to be ...,disappointment,disappointment
3,"I'm just going to be honest, nobody was happy ...",disapproval,disapproval
4,Hi everyone. I'm a 36 year old male and have b...,caring,caring
5,"When I graduated high school in 2020, my paren...",confusion,remorse
6,I (15f) have a bf (15m) Today I was gonna hang...,remorse,caring
7,"Before my brother went to jail, he asked me to...",caring,annoyance
8,"Over a month ago, my (28F) girlfriend (27F) we...",sadness,sadness
9,"Alright, so me 18(f) and my older sister, 20s(...",annoyance,annoyance


In [30]:
classifier_labels = []
for text in to_label["text"]:
    prediction = classifier(text)[0]
    top_emotion = max(prediction, key=lambda x: x["score"])["label"]
    classifier_labels.append(top_emotion)

to_label["classifier_label"] = classifier_labels

print("Manual vs LLM agreement:", (to_label["manual_label"] == to_label["llm_label"]).mean())
print("Manual vs Classifier agreement:", (to_label["manual_label"] == to_label["classifier_label"]).mean())
print("LLM vs Classifier agreement:", (to_label["llm_label"] == to_label["classifier_label"]).mean())

to_label[["text", "manual_label", "llm_label", "classifier_label"]]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Manual vs LLM agreement: 0.7333333333333333
Manual vs Classifier agreement: 0.2
LLM vs Classifier agreement: 0.13333333333333333


,text,manual_label,llm_label,classifier_label
0,So over the past few months I’ve been living w...,annoyance,annoyance,neutral
1,"&#x200B;\n\n... I'm tired of this female, to b...",anger,anger,annoyance
2,My daughter’s school did not choose her to be ...,disappointment,disappointment,annoyance
3,"I'm just going to be honest, nobody was happy ...",disapproval,disapproval,realization
4,Hi everyone. I'm a 36 year old male and have b...,caring,caring,neutral
5,"When I graduated high school in 2020, my paren...",confusion,remorse,realization
6,I (15f) have a bf (15m) Today I was gonna hang...,remorse,caring,remorse
7,"Before my brother went to jail, he asked me to...",caring,annoyance,neutral
8,"Over a month ago, my (28F) girlfriend (27F) we...",sadness,sadness,sadness
9,"Alright, so me 18(f) and my older sister, 20s(...",annoyance,annoyance,annoyance


In [31]:
# ===== GoEmotions full-test-set evaluation (thresholds 0.3 and 0.5) =====
!pip -q install datasets transformers scikit-learn pandas

import numpy as np, pandas as pd, torch
from datasets import load_dataset
from sklearn.metrics import f1_score, precision_recall_fscore_support
from transformers import pipeline

MODEL = "SamLowe/roberta-base-go_emotions"
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
test = dataset["test"]
label_names = test.features["labels"].feature.names   # 28 labels incl. neutral
L = len(label_names)
print("Test examples:", len(test), "| labels:", L)

device = 0 if torch.cuda.is_available() else -1        # use GPU runtime if you can
clf = pipeline("text-classification", model=MODEL, top_k=None,
               truncation=True, max_length=128, device=device)

texts = list(test["text"])  # Force cast to a standard list of strings
probs = np.zeros((len(texts), L), dtype=np.float32)
idx = {n: i for i, n in enumerate(label_names)}

out = clf(texts, batch_size=64)
for r, preds in enumerate(out):
    for p in preds:
        probs[r, idx[p["label"]]] = p["score"]          # raises KeyError if label strings differ

y_true = np.zeros((len(texts), L), dtype=int)
for r, labs in enumerate(test["labels"]):
    y_true[r, labs] = 1

np.save("probs_full_test.npy", probs)                    # reuse later without re-running the model

def evaluate(th):
    y_pred = (probs > th).astype(int)                    # authors use ">" (strict) too
    macro_all = f1_score(y_true, y_pred, average="macro", zero_division=0)
    keep = [i for i, n in enumerate(label_names) if n != "neutral"]
    macro_27 = f1_score(y_true[:, keep], y_pred[:, keep], average="macro", zero_division=0)
    micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    return y_pred, macro_all, macro_27, micro

rows = {}
for th in (0.3, 0.5):
    y_pred, m28, m27, mi = evaluate(th)
    print(f"threshold {th}: macro-F1 (28 labels)={m28:.4f} | macro-F1 (27 emotions, no neutral)={m27:.4f} | micro-F1={mi:.4f}")
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, zero_division=0)
    rows[th] = pd.DataFrame({"emotion": label_names, "support": s,
                             f"P@{th}": p, f"R@{th}": r, f"F1@{th}": f})

per_class = rows[0.3].merge(rows[0.5].drop(columns="support"), on="emotion")
per_class = per_class.sort_values("support", ascending=False)
per_class.to_csv("per_emotion_f1_full_test.csv", index=False)
print(per_class.round(3).to_string(index=False))
print("\nLowest-F1 emotions @0.3:")
print(per_class.nsmallest(5, "F1@0.3")[["emotion", "support", "F1@0.3"]].round(3).to_string(index=False))

Test examples: 5427 | labels: 28


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

threshold 0.3: macro-F1 (28 labels)=0.4986 | macro-F1 (27 emotions, no neutral)=0.4916 | micro-F1=0.6086
threshold 0.5: macro-F1 (28 labels)=0.4503 | macro-F1 (27 emotions, no neutral)=0.4431 | micro-F1=0.5855
       emotion  support  P@0.3  R@0.3  F1@0.3  P@0.5  R@0.5  F1@0.5
       neutral     1787  0.622  0.769   0.688  0.694  0.604   0.646
    admiration      504  0.673  0.744   0.707  0.725  0.675   0.699
     gratitude      352  0.935  0.901   0.918  0.960  0.881   0.919
      approval      351  0.432  0.442   0.437  0.609  0.302   0.404
     annoyance      320  0.363  0.328   0.345  0.472  0.159   0.238
     curiosity      284  0.484  0.648   0.554  0.537  0.356   0.428
   disapproval      267  0.414  0.468   0.439  0.494  0.307   0.379
     amusement      264  0.746  0.924   0.826  0.790  0.871   0.829
          love      238  0.746  0.887   0.810  0.773  0.832   0.802
         anger      198  0.536  0.485   0.509  0.652  0.379   0.479
      optimism      186  0.608  0.484   0.

In [32]:
# ===== Agreement analysis: manual vs LLM-judge vs classifier =====
!pip -q install scikit-learn pandas statsmodels

import pandas as pd, numpy as np, itertools
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# ---------- 1. CONFIG: edit these to match YOUR file ----------
CSV_PATH   = "manually_labeled_sample.csv"   # your actual labelled file
TEXT_COL   = "text"
MANUAL_COL = "manual_label"       # your own labels
LLM_COL    = "llm_label"          # LLM-judge labels
CLF_COL    = "classifier_label"   # classifier top-1 emotion
# If a cell can hold several labels, they may be separated by | or , (first = primary)
# --------------------------------------------------------------

# Re-integrate the labels from active kernel memory if missing from disk
df = pd.read_csv(CSV_PATH)
if LLM_COL not in df.columns and 'llm_labels' in globals():
    df[LLM_COL] = llm_labels
if CLF_COL not in df.columns and 'classifier_labels' in globals():
    df[CLF_COL] = classifier_labels

# Save back so the file is complete on disk
df.to_csv(CSV_PATH, index=False)

print("Columns found:", list(df.columns))
missing = [c for c in (TEXT_COL, MANUAL_COL, LLM_COL, CLF_COL) if c not in df.columns]
assert not missing, f"Rename these in the CONFIG block, not found in CSV: {missing}"

def split_labels(x):
    if pd.isna(x): return []
    return [t.strip().lower() for t in str(x).replace("|", ",").split(",") if t.strip()]

for c in (MANUAL_COL, LLM_COL, CLF_COL):
    df[c + "_set"] = df[c].apply(split_labels)
    df[c + "_primary"] = df[c + "_set"].apply(lambda s: s[0] if s else np.nan)

# Keep only rows labelled by all three sources
d = df.dropna(subset=[MANUAL_COL + "_primary", LLM_COL + "_primary", CLF_COL + "_primary"]).copy()
print(f"\nItems with all three labels: {len(d)} of {len(df)}")
if len(d) < 100:
    print("WARNING: fewer than 100 items; kappa will be noisy. Report this honestly.")

# Validate label vocabulary against the 28 GoEmotions labels
VALID = {"admiration","amusement","anger","annoyance","approval","caring","confusion","curiosity",
         "desire","disappointment","disapproval","disgust","embarrassment","excitement","fear",
         "gratitude","grief","joy","love","nervousness","optimism","pride","realization","relief",
         "remorse","sadness","surprise","neutral"}
for c in (MANUAL_COL, LLM_COL, CLF_COL):
    bad = set(d[c + "_primary"]) - VALID
    if bad: print(f"WARNING: non-GoEmotions labels in {c}: {bad}  -> fix or map them")

# ---------- 2. Cohen's kappa (primary label, 28-way) ----------
pairs = [(MANUAL_COL, LLM_COL, "Manual vs LLM"),
         (MANUAL_COL, CLF_COL, "Manual vs Classifier"),
         (LLM_COL,    CLF_COL, "LLM vs Classifier")]

def boot_ci(a, b, n=2000, seed=0):
    rng = np.random.default_rng(seed); a = np.array(a); b = np.array(b); ks = []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a))
        ks.append(cohen_kappa_score(a[i], b[i]))
    return np.nanpercentile(ks, [2.5, 97.5])

rows = []
for x, y, name in pairs:
    a, b = d[x + "_primary"], d[y + "_primary"]
    k = cohen_kappa_score(a, b)
    lo, hi = boot_ci(a, b)
    exact = (a.values == b.values).mean()
    overlap = np.mean([len(set(s1) & set(s2)) > 0 for s1, s2 in zip(d[x + "_set"], d[y + "_set"])])
    rows.append({"pair": name, "exact_agree": exact, "any_label_overlap": overlap,
                 "cohen_kappa": k, "kappa_95CI_low": lo, "kappa_95CI_high": hi})
res = pd.DataFrame(rows)
print("\n=== 28-way agreement (primary label) ===")
print(res.round(3).to_string(index=False))

# ---------- 3. Coarse agreement: sentiment groups (less noisy with ~100 items) ----------
GROUP = {}
for e in ["admiration","amusement","approval","caring","desire","excitement","gratitude","joy",
          "love","optimism","pride","relief"]: GROUP[e] = "positive"
for e in ["anger","annoyance","disappointment","disapproval","disgust","embarrassment","fear",
          "grief","nervousness","remorse","sadness"]: GROUP[e] = "negative"
for e in ["confusion","curiosity","realization","surprise"]: GROUP[e] = "ambiguous"
GROUP["neutral"] = "neutral"

for c in (MANUAL_COL, LLM_COL, CLF_COL):
    d[c + "_grp"] = d[c + "_primary"].map(GROUP)
rows = []
for x, y, name in pairs:
    a, b = d[x + "_grp"], d[y + "_grp"]
    ok = a.notna() & b.notna()
    rows.append({"pair": name, "n": int(ok.sum()),
                 "exact_agree": (a[ok] == b[ok]).mean(),
                 "cohen_kappa": cohen_kappa_score(a[ok], b[ok])})
print("\n=== 4-way sentiment-group agreement (positive/negative/ambiguous/neutral) ===")
print(pd.DataFrame(rows).round(3).to_string(index=False))

# ---------- 4. Three-rater agreement (Fleiss) ----------
try:
    from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
    cats = sorted(VALID)
    m = d[[MANUAL_COL + "_primary", LLM_COL + "_primary", CLF_COL + "_primary"]].apply(
        lambda col: col.map({c: i for i, c in enumerate(cats)})).values
    table, _ = aggregate_raters(m)
    print(f"\nFleiss kappa (manual, LLM, classifier; 28-way): {fleiss_kappa(table):.3f}")
except Exception as e:
    print("Fleiss kappa skipped:", e)

# ---------- 5. Per-emotion agreement (manual as reference) ----------
per = []
for e in sorted(set(d[MANUAL_COL + "_primary"])):
    sub = d[d[MANUAL_COL + "_primary"] == e]
    per.append({"emotion (manual)": e, "n": len(sub),
                "LLM_agree": (sub[LLM_COL + "_primary"] == e).mean(),
                "Classifier_agree": (sub[CLF_COL + "_primary"] == e).mean()})
per = pd.DataFrame(per).sort_values("n", ascending=False)
print("\n=== Per-emotion agreement with MANUAL labels ===")
print(per.round(2).to_string(index=False))

# ---------- 6. Disagreements for the bias / ambiguity probe ----------
d["all_agree"] = (d[MANUAL_COL + "_primary"] == d[LLM_COL + "_primary"]) & \
                 (d[MANUAL_COL + "_primary"] == d[CLF_COL + "_primary"])
d["n_distinct"] = d[[MANUAL_COL + "_primary", LLM_COL + "_primary", CLF_COL + "_primary"]].nunique(axis=1)
dis = d[~d["all_agree"]].sort_values("n_distinct", ascending=False)
print(f"\nAll three agree on {d['all_agree'].mean():.0%} of items; "
      f"all three differ on {(d['n_distinct']==3).mean():.0%}.")
pd.set_option("display.max_colwidth", 140)
print("\nMost contested items (inspect for sarcasm / cultural context / ambiguity):")
print(dis[[TEXT_COL, MANUAL_COL + "_primary", LLM_COL + "_primary", CLF_COL + "_primary"]].head(15).to_string(index=False))

# ---------- 7. Save outputs for the repo ----------
res.to_csv("agreement_kappa_28way.csv", index=False)
per.to_csv("agreement_per_emotion.csv", index=False)
dis.drop(columns=[c for c in d.columns if c.endswith("_set")]).to_csv("contested_items.csv", index=False)
print("\nSaved: agreement_kappa_28way.csv, agreement_per_emotion.csv, contested_items.csv")

Columns found: ['text', 'date_utc', 'score', 'subreddit', 'word_count', 'manual_label', 'llm_label', 'classifier_label']

Items with all three labels: 15 of 15

=== 28-way agreement (primary label) ===
                pair  exact_agree  any_label_overlap  cohen_kappa  kappa_95CI_low  kappa_95CI_high
       Manual vs LLM        0.733              0.733        0.695           0.413            0.919
Manual vs Classifier        0.200              0.200        0.147          -0.037            0.341
   LLM vs Classifier        0.133              0.133        0.058          -0.098            0.235

=== 4-way sentiment-group agreement (positive/negative/ambiguous/neutral) ===
                pair  n  exact_agree  cohen_kappa
       Manual vs LLM 14        0.786        0.455
Manual vs Classifier 14        0.571        0.359
   LLM vs Classifier 15        0.400        0.100

Fleiss kappa (manual, LLM, classifier; 28-way): 0.269

=== Per-emotion agreement with MANUAL labels ===
emotion (manual)  